# VC ノードを起動する

VCP SDK を使って VC ノードを 1 台起動し、ログインして動作を確認したうえで削除します。VCP の基本的な操作を一通り体験できます。

所要時間は 15 分程度です。うち VC ノードの起動に 2〜3 分かかります。

**前提**

- VC コントローラが構築済みであること
- VC コントローラのアクセストークンを入手していること
- 使用するクラウドプロバイダの認証情報が VC コントローラに登録されていること

> **注意**
>
> VC ノードはクラウド上の計算資源を使用します。**確認が終わったら、最後の「6. VC ノードの削除」を必ず実行してください。** 起動したままにすると、クラウドプロバイダの利用料金が発生し続けます。

## 1. 設定

VC ノードを起動するために必要な情報を設定します。このセクションのセルは、上から順に実行してください。

### アクセストークンの入力

VC コントローラを操作するには、アクセストークンが必要です。

次のセルを実行すると入力欄が表示されます。アクセストークンを入力し、Enter キーで確定してください。入力した値は画面に表示されません。

> アクセストークンを持っていない場合は、VC 管理者に発行を依頼してください。

In [ ]:
from getpass import getpass

vcc_access_token = getpass()

### SSH 鍵の確認

起動した VC ノードには SSH でログインします。ログインに使用する鍵をこの環境に用意します。

次のセルは、鍵がまだ無い場合にのみ新しく作成します。すでに `~/.ssh/id_ed25519` がある場合は何もしません。

In [ ]:
!test -f ~/.ssh/id_ed25519 || ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N ""
!ls -l ~/.ssh/id_ed25519 ~/.ssh/id_ed25519.pub

### 起動する VC ノードの指定

VC ノードをどのクラウドに、どのアドレスで起動するかを指定します。以下の 3 つは**必ず自分の環境に合わせて指定してください。**

| 変数 | 内容 |
|---|---|
| `provider` | 使用するクラウドプロバイダ |
| `vcnode_ipaddress` | VC ノードに割り当てるプライベート IP アドレス |
| `ssh_username` | クラウドの仮想マシンにログインするユーザ名 |

`provider` に指定できる値は `aws`、`aws_spot`、`azure`、`gcp`、`sakura`、`proxmox`、`mdx2`、`oracle`、`onpremises` などです。指定できるプロバイダの詳細は SDK リファレンスを参照してください。

`vcnode_ipaddress` には、VC コントローラに定義したクラウド仮想ネットワークの範囲内のアドレスを指定してください。範囲外のアドレスを指定すると起動に失敗します。

`ssh_username` はクラウドの仮想マシン側のログインユーザです。オンプレミス環境などで `ubuntu` 以外の場合は変更してください。ベースコンテナ内のユーザとは別のものである点に注意してください。

In [ ]:
# 使用するクラウドプロバイダ
provider = 

# VC ノードに割り当てるプライベート IP アドレス
vcnode_ipaddress = '192.168.100.70'

# クラウドの仮想マシンにログインするユーザ名
ssh_username = 'ubuntu'

### その他の設定

以下は既定のままで動作します。**通常は変更する必要はありません。**

| 変数 | 内容 |
|---|---|
| `flavor` | 計算資源の大きさ |
| `ug_name` | 作成する UnitGroup の名前。VC ノードをまとめる単位です |
| `bc_image_name` | VC ノード上で動作させるベースコンテナのイメージ |
| `service_manager_cmd` | ベースコンテナ内のプロセス管理コマンド。動作確認で使用します |
| `bc_user` | ベースコンテナ内のユーザ名。動作確認の SSH ログインで使用します |
| `gpu` | GPU を使用するかどうか |

GPU を利用する場合は、`gpu` を `True` にしたうえで、`flavor` と `bc_image_name` を GPU 向けのものに変更してください。

```python
gpu = True
flavor = 'gpu'
bc_image_name = 'vcp/base:3.0.0-ubuntu24.04-gpu-dev'
```

さらに、**AWS 以外のプロバイダで GPU を利用する場合は、起動処理 (`deploy_vcnode()`) の中で `spec.cloud_image` に NVIDIA ドライバがインストール済みのマシンイメージを指定する必要があります。** GPU 版のベースコンテナは、ホストとなる仮想マシンに NVIDIA ドライバが導入されていることを前提としているためです。AWS では GPU 対応の AMI があらかじめ指定されています。

In [ ]:
flavor = 'medium'
ug_name = 'unitgroup_example'
bc_image_name = 'vcp/base:3.0.0-alpine3.22-dev'
service_manager_cmd = 'supervisorctl'
bc_user = 'root'
gpu = False

## 2. VCP SDK の初期化

VCP SDK を読み込み、VC コントローラに接続します。

アクセストークンが正しくない場合、次のセルはエラーになります。その場合は「アクセストークンの入力」からやり直してください。

In [ ]:
from vcpsdk.vcpsdk import VcpSDK

vcp = VcpSDK(vcc_access_token)

## 3. 起動処理の定義

VC ノードを起動する処理を関数として定義します。次のセルを実行して、この Notebook に読み込んでください。この段階ではまだ VC ノードは起動しません。

関数 `deploy_vcnode()` は次の処理を行います。

1. 同じ名前の UnitGroup が残っている場合は削除する
2. UnitGroup を新しく作成する
3. 「1. 設定」で指定した内容から、VC ノードの構成 (`spec`) を組み立てる
4. Unit を作成し、VC ノードを起動する

> **注意**
>
> 手順 1 により、`ug_name` と同じ名前の UnitGroup がすでに存在する場合、その UnitGroup と配下の VC ノードは削除されます。使用中の UnitGroup と名前が重複しないよう注意してください。

In [ ]:
import os
import time

from IPython.display import display, HTML


def deploy_vcnode(vcp, ip_address=None, ssh_username=None, disks: list = []):
    # 同名の UnitGroup が残っている場合は削除する
    try:
        ugroup = vcp.get_ugroup(ug_name)
        if ugroup is not None:
            ugroup.cleanup()
            while vcp.get_ugroup(ug_name):
                time.sleep(5)
    except Exception:
        pass

    # UnitGroup の作成
    ugroup = vcp.create_ugroup(ug_name, 'compute')

    # VC ノードの構成を組み立てる
    spec = vcp.get_spec(provider, flavor)

    if provider == 'onpremises':
        spec.user_name = ssh_username

    # プライベート IP アドレス
    if ip_address is not None and len(ip_address) > 0:
        spec.ip_addresses = [ip_address]

    # ベースコンテナのイメージ
    spec.image = bc_image_name

    # ベースコンテナに登録する SSH 公開鍵
    spec.set_ssh_pubkey(os.path.expanduser('~/.ssh/id_ed25519.pub'))

    # VC ノードの起動直後に実行するコマンド
    spec.user_init_command = """#! /bin/bash
    echo creating test file
    touch /root/user_init_command.test
    echo created test file
    """

    if gpu:
        spec.gpus = 'all'
        # GPU 版ベースコンテナには NVIDIA ドライバ導入済みのマシンイメージが必要
        if provider == 'aws':
            spec.cloud_image = 'ami-0d0132b10bb7e1623'
        # AWS 以外の場合は、プロバイダに応じたイメージを指定する
        # spec.cloud_image = '...'

    if len(disks) > 0:
        spec.disks = disks

    # Unit の作成と VC ノードの起動
    ugroup.create_unit(f'{ug_name}_exampleunit', spec)


def print_styled_df(styled_df):
    display(HTML(styled_df.to_html()))

## 4. VC ノードの起動

定義した処理を実行して、UnitGroup の作成と VC ノードの起動を行います。

**起動には 2〜3 分かかります。** 実行中は `BOOTING ... N sec` のようなログが表示されます。完了すると、作成された Unit と VC ノードの一覧が表形式で表示されます。VC ノードの `node_state` が `RUNNING` になっていることを確認してください。

In [ ]:
deploy_vcnode(vcp, vcnode_ipaddress, ssh_username=ssh_username)

ug = vcp.get_ugroup(ug_name)
print_styled_df(ug.df_units())

unit = ug.get_unit(f'{ug_name}_exampleunit')
print_styled_df(unit.df_nodes())

## 5. 動作の確認

起動した VC ノードに SSH でログインし、ベースコンテナが動作していることを確認します。

ログイン先はクラウドの仮想マシンではなく、その上で動作しているベースコンテナです。まず OS の情報を表示して、指定したイメージのベースコンテナが動作していることを確認します。

In [ ]:
ssh_opts = '-o StrictHostKeyChecking=no'

!ssh {ssh_opts} {bc_user}@{vcnode_ipaddress} cat /etc/os-release

次に、ベースコンテナ内で動作しているサービスの状態を確認します。死活監視やメトリクス収集などのプロセスが動作しています。

In [ ]:
!ssh {ssh_opts} {bc_user}@{vcnode_ipaddress} {service_manager_cmd} status

## 6. VC ノードの削除

起動した VC ノードを、UnitGroup ごと削除します。

**このセルの実行を忘れると、クラウドプロバイダの利用料金が発生し続けます。** 動作確認が終わったら必ず実行してください。

削除の完了後、`cleanup completed` と表示されます。

In [ ]:
ugroup = vcp.get_ugroup(ug_name)
ugroup.cleanup()

以上で、VC ノードの起動から削除までの一連の操作は完了です。

GPU の利用や複数のクラウドプロバイダを組み合わせた構成など、応用的な使い方についてはチュートリアルを参照してください。